## 1D Invasion Metric Trend

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
from matplotlib.ticker import MaxNLocator

# Load data
file_path = "../../Results/Data/invasion_metrics.csv"
df = pd.read_csv(file_path)

# Define metrics and colors
metrics = ["Invasive Area", "Infiltrative Area", "Singles", "Fingers", "Clusters"]
custom_colors = {
    "Invasive Area": "red",
    "Infiltrative Area": "orange",
    "Singles": "blue",
    "Fingers": "green",
    "Clusters": "magenta"
}

# Parameter definitions (no fixing — overall averages)
param_list   = ["Proliferative Probability", "Contact Energy", "Chemotaxis Lambda"]
param_labels = ["Proliferative Probability", "Contact Energy", "Chemotaxis Lambda"]

# Output filenames (keep special label for Chemotaxis Lambda)
custom_filename_prefix = {
    "Proliferative Probability": "proliferative_probability_vs_invasion_metrics_overall",
    "Contact Energy": "contact_energy_vs_invasion_metrics_overall",
    "Chemotaxis Lambda": "migration_coefficient_vs_invasion_metrics_overall",
}

# ---------- Helpers ----------
def safe_groupby_mean_sem(data: pd.DataFrame, by: str, metric: str) -> pd.DataFrame:
    """Group by 'by' and compute mean & sem for a metric; returns sorted by 'by'."""
    grouped = (
        data.groupby(by)[metric]
        .agg(["mean", "sem"])
        .reset_index()
        .sort_values(by)
    )
    # If sem is NaN (e.g., single observation), replace with 0 so bands render
    grouped["sem"] = grouped["sem"].fillna(0.0)
    # Rename for consistency
    grouped.columns = [by, f"{metric}_mean", f"{metric}_sem"]
    return grouped

def compute_global_y_limits_overall(df: pd.DataFrame, metrics: list, params: list) -> dict:
    """Global y-limits across all parameter sweeps, using overall (unfixed) averages."""
    y_limits = {}
    for metric in metrics:
        all_vals = []
        for param in params:
            g = safe_groupby_mean_sem(df, param, metric)
            means = g[f"{metric}_mean"].to_numpy()
            sems  = g[f"{metric}_sem"].to_numpy()
            all_vals.extend(list(means + sems))
            all_vals.extend(list(means - sems))
        if len(all_vals) == 0:
            y_limits[metric] = (0, 1)
            continue
        min_y = float(np.nanmin(all_vals))
        max_y = float(np.nanmax(all_vals))
        if not np.isfinite(min_y) or not np.isfinite(max_y):
            y_limits[metric] = (0, 1)
            continue
        if max_y == min_y:
            # Degenerate; give a small buffer
            min_y, max_y = min_y - 0.5, max_y + 0.5
        buffer = 0.05 * (max_y - min_y)
        y_limits[metric] = (min_y - buffer, max_y + buffer)
    return y_limits

global_y_limits = compute_global_y_limits_overall(df, metrics, param_list)

# Create individual figures per parameter (overall averages)
for (param, param_label) in zip(param_list, param_labels):
    # Prepare x and grouped stats for each metric
    # Use the union x-grid across the param (sorted unique)
    x_unique = np.sort(df[param].dropna().unique())
    if x_unique.size == 0:
        # Nothing to plot for this parameter
        continue

    # Create figure with 5 rows (one per metric)
    fig, axes = plt.subplots(len(metrics), 1, figsize=(6, 15), sharex=True)

    for row_idx, metric in enumerate(metrics):
        ax = axes[row_idx]
        g = safe_groupby_mean_sem(df, param, metric)

        # Align to x_unique to keep consistent x across metrics
        # (not strictly necessary, but helps align the vertical grid)
        g = g.set_index(param).reindex(x_unique).reset_index()
        x = g[param].to_numpy()
        y = g[f"{metric}_mean"].to_numpy()
        yerr = g[f"{metric}_sem"].to_numpy()
        color = custom_colors[metric]

        # Spline only if we have enough unique, finite points
        mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(yerr)
        x_plot = x[mask]
        y_plot = y[mask]
        e_plot = yerr[mask]

        if x_plot.size >= 4 and np.unique(x_plot).size >= 4:
            # Avoid issues with non-strictly increasing: sort
            order = np.argsort(x_plot)
            x_plot = x_plot[order]
            y_plot = y_plot[order]
            e_plot = e_plot[order]

            x_smooth = np.linspace(x_plot.min(), x_plot.max(), 300)
            y_spline = make_interp_spline(x_plot, y_plot, k=3)(x_smooth)
            err_spline = make_interp_spline(x_plot, e_plot, k=3)(x_smooth)
        else:
            x_smooth, y_spline, err_spline = x_plot, y_plot, e_plot

        # Plot line and SEM band
        ax.plot(x_smooth, y_spline, color=color)
        ax.fill_between(x_smooth, y_spline - err_spline, y_spline + err_spline,
                        color=color, alpha=0.2)

        # Y-limits shared across all sweeps
        ax.set_ylim(global_y_limits[metric])

        # Styling
        ax.grid(False)
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_ylabel(metric, fontsize=18, weight='bold', color=color)

        # Always show x-tick labels (since sharex=True)
        ax.tick_params(axis='both', labelsize=14, width=1.5, labelbottom=True)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontweight('bold')

        # Invert x-axis for Contact Energy (if you want to keep that convention)
        if param == "Contact Energy":
            ax.invert_xaxis()

    # X label text (rename Chemotaxis Lambda to Migration Coefficient)
    xlabel_text = (
        "Migration Coefficient" if param == "Chemotaxis Lambda"
        else "Contact Energy (Inverted)" if param == "Contact Energy"
        else param_label
    )
    axes[-1].set_xlabel(xlabel_text, fontsize=18, weight='bold')

    # Save
    fig.tight_layout()
    outname = f"{custom_filename_prefix[param]}.png"
    save_path = os.path.join(
        "../../Results/Plots/Invasion_Metrics_Plots/Invasion_Metrics_Trends", outname)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300)
    plt.close(fig)


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import os


base_path = "../../Results/Plots/Invasion_Metrics_Plots/Invasion_Metrics_Trends/"
img_A = Image.open(os.path.join(base_path, "migration_coefficient_vs_invasion_metrics.png"))           # A
img_B = Image.open(os.path.join(base_path, "contact_energy_vs_invasion_metrics.png"))              # B
img_C = Image.open(os.path.join(base_path, "proliferative_probability_vs_invasion_metrics.png"))   # C

max_height = max(img_A.height, img_B.height, img_C.height)
target_height = max_height

def resize_to_height(img, target_height):
    scale = target_height / img.height
    new_width = int(img.width * scale)
    return img.resize((new_width, target_height))

img_A_resized = resize_to_height(img_A, target_height)
img_B_resized = resize_to_height(img_B, target_height)
img_C_resized = resize_to_height(img_C, target_height)


total_width = img_A_resized.width + img_B_resized.width + img_C_resized.width
combined_img = Image.new("RGB", (total_width, target_height))
combined_img.paste(img_A_resized, (0, 0))
combined_img.paste(img_B_resized, (img_A_resized.width, 0))
combined_img.paste(img_C_resized, (img_A_resized.width + img_B_resized.width, 0))

# Display with labels
plt.figure(figsize=(total_width / 100, target_height / 100))
plt.imshow(combined_img)
plt.axis('off')

# Add bold lettered labels (A), (B), (C)
plt.text(10, 30, "A", fontsize=72, fontweight='bold', color='black', backgroundcolor='white')
plt.text(img_A_resized.width + 10, 30, "B", fontsize=72, fontweight='bold', color='black', backgroundcolor='white')
plt.text(img_A_resized.width + img_B_resized.width + 10, 30, "C", fontsize=72, fontweight='bold', color='black', backgroundcolor='white')


final_save_path = "../../Results/Plots/parameters_vs_invasion_metrics.png"
plt.savefig(final_save_path, dpi=300, bbox_inches='tight')
plt.show()


In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy.interpolate import make_interp_spline
from matplotlib.ticker import MaxNLocator
from scipy.stats import f_oneway, kruskal
from statsmodels.stats.multitest import multipletests
from pathlib import Path
from matplotlib.lines import Line2D


# -----------------------------
# Load data
# -----------------------------
file_path = "../../Results/Data/invasion_metrics.csv"
df = pd.read_csv(file_path)


# Which metrics should show legends?
INCLUDE_LEGEND_FOR = {
    # "Invasive Area",
    # "Infiltrative Area",
    # "Singles",
    # "Fingers",
    # "Clusters",
}
# Legend style (optional)
LEGEND_KW = dict(frameon=False, fontsize=11, loc="best")


# -----------------------------
# Invasion metrics & base colors
# -----------------------------
metrics = ["Invasive Area", "Infiltrative Area", "Singles", "Fingers", "Clusters"]
base_colors = {
    "Invasive Area":      "red",
    "Infiltrative Area":  "purple",
    "Singles":            "blue",
    "Fingers":            "green",
    "Clusters":           "magenta",
}

# Parameters
params = ["Proliferative Probability", "Contact Energy", "Chemotaxis Lambda"]
param_labels_for_legend = {
    "Proliferative Probability": "Proliferative Probability",
    "Contact Energy": "Contact Energy",
    "Chemotaxis Lambda": "Migration Coefficient", 
}

# -----------------------------
# Output dirs
# -----------------------------
out_dir = Path("../../Results/Plots/Invasion_Metrics_Plots/Invasion_Metrics_Trends")
out_dir.mkdir(parents=True, exist_ok=True)
stats_dir = out_dir / "stats"
stats_dir.mkdir(parents=True, exist_ok=True)

summary_txt_path = stats_dir / "figure_stats_summary.txt"

# -----------------------------
# Normalize parameters to [0,1]
# -----------------------------
norm_info = []
for p in params:
    p_min, p_max = df[p].min(), df[p].max()
    if p_max > p_min:
        df[p + "_norm"] = (df[p] - p_min) / (p_max - p_min)
    else:
        df[p + "_norm"] = 0.0 
    norm_info.append((p, float(p_min), float(p_max)))

# -----------------------------
# Utility: 
# -----------------------------
def make_shades(color):
    r, g, b = to_rgb(color)
    shades = {
        "Proliferative Probability": (min(r+0.6,1), min(g+0.6,1), min(b+0.6,1)),  # light
        "Contact Energy": (r, g, b),                                             # base
        "Chemotaxis Lambda": (r*0.4, g*0.4, b*0.4),                              # dark
    }
    return shades

# Distinct line styles for B/W friendliness
line_styles = {
    "Proliferative Probability": "--",  # dashed
    "Contact Energy": "-",              # solid
    "Chemotaxis Lambda": "-.",          # dash-dot
}

# -----------------------------
# Helpers
# -----------------------------
def group_mean_sem(data: pd.DataFrame, by: str, metric: str) -> pd.DataFrame:
    """
    Group by 'by' and compute n, mean, std, sem for 'metric'; sorted by 'by'.
    SEM uses pandas sem() (ddof=1). We also include std for auditing.
    """
    g = (
        data.groupby(by)[metric]
        .agg(n="count", mean="mean", std="std", sem="sem")
        .reset_index()
        .sort_values(by)
    )
    # Replace NaN (e.g., single obs) SEM with 0 for plotting bands
    g["sem"] = g["sem"].fillna(0.0)
    # Reorder/rename for clarity
    g = g[[by, "n", "mean", "std", "sem"]]
    return g

def compute_global_y_limits(df, metrics, params):
    """
    Global y-limits across all metrics & parameters based on mean±sem of the normalized sweeps.
    """
    ylims = {}
    for metric in metrics:
        vals = []
        for p in params:
            pn = p + "_norm"
            g = group_mean_sem(df, pn, metric)
            m, s = g["mean"].to_numpy(), g["sem"].to_numpy()
            vals.extend(list(m + s))
            vals.extend(list(m - s))
        if len(vals) == 0:
            ylims[metric] = (0, 1)
            continue
        vmin, vmax = np.nanmin(vals), np.nanmax(vals)
        if not np.isfinite(vmin) or not np.isfinite(vmax):
            ylims[metric] = (0, 1)
            continue
        if vmax == vmin:
            vmin, vmax = vmin - 0.5, vmax + 0.5
        pad = 0.05 * (vmax - vmin)
        ylims[metric] = (vmin - pad, vmax + pad)
    return ylims

global_y_limits = compute_global_y_limits(df, metrics, params)

# -----------------------------
# Per-bin p-values across parameters (PP, CE, MC)
# -----------------------------
def compute_per_bin_pvalues(df, metric_name, num_bins=10):

    # Long-format assembly
    long_parts = []
    mapping = [("Proliferative Probability", "PP"),
               ("Contact Energy", "CE"),
               ("Chemotaxis Lambda", "MC")]
    for col, label in mapping:
        pn = col + "_norm"
        tmp = df[[pn, metric_name]].copy()
        tmp = tmp.rename(columns={pn: "param_norm", metric_name: "metric_value"})
        tmp["param_type"] = label
        long_parts.append(tmp)
    LONG = pd.concat(long_parts, ignore_index=True)

    # Bin the normalized parameter
    LONG["bin"] = pd.cut(
        LONG["param_norm"],
        bins=np.linspace(0, 1, num_bins + 1),
        include_lowest=True
    )

    rows = []
    for b, sub in LONG.groupby("bin", observed=True):
        groups = []
        sizes = {}
        for label in ["PP", "CE", "MC"]:
            vals = sub.loc[sub["param_type"] == label, "metric_value"].dropna().to_numpy()
            groups.append(vals)
            sizes[label] = int(len(vals))
        if sum(len(g) >= 2 for g in groups) >= 2:
            try:
                stat, p = f_oneway(*groups)
                test_used = "ANOVA"
            except Exception:
                stat, p = kruskal(*groups)
                test_used = "Kruskal"
        else:
            stat, p, test_used = np.nan, np.nan, "NA"
        rows.append({
            "bin": str(b),
            "test": test_used,
            "stat": stat,
            "p_raw": p,
            "n_PP": sizes.get("PP", 0),
            "n_CE": sizes.get("CE", 0),
            "n_MC": sizes.get("MC", 0),
        })

    res = pd.DataFrame(rows)
    # FDR correction across bins
    mask = res["p_raw"].notna()
    if mask.any():
        _, p_fdr, _, _ = multipletests(res.loc[mask, "p_raw"], method="fdr_bh")
        res.loc[mask, "p_fdr"] = p_fdr
    else:
        res["p_fdr"] = np.nan
    return res

summary_lines = []
summary_lines.append("Figure statistics & calculations summary")
summary_lines.append(f"Source file: {file_path}")
summary_lines.append("Normalization to [0,1] was applied independently per parameter using min/max from data.")
summary_lines.append("Parameters & normalization ranges (original scale):")
for name, vmin, vmax in norm_info:
    summary_lines.append(f"  - {name}: min={vmin:.6g}, max={vmax:.6g}")
summary_lines.append("")
summary_lines.append("Mean ± SEM computation:")
summary_lines.append("  - For each unique normalized parameter value, observations of the metric")
summary_lines.append("    were aggregated: n = count, mean = average, SEM = std/sqrt(n) with ddof=1.")
summary_lines.append("  - Visualization uses cubic spline smoothing for the mean and SEM;")
summary_lines.append("    statistics reported in CSVs are computed on the unsmoothed groups.")
summary_lines.append("")

for metric in metrics:
    fig, ax = plt.subplots(figsize=(7, 5))
    shades = make_shades(base_colors[metric])

    n_values_all = []

    for p in params:
        pn = p + "_norm"
        g = group_mean_sem(df, pn, metric)  # includes n, mean, std, sem
        
        audit_csv = stats_dir / f"audit_{metric.replace(' ', '_').lower()}_by_{pn}.csv"
        g.to_csv(audit_csv, index=False)

        g["_sem_check"] = g["std"] / np.sqrt(g["n"].clip(lower=1))
        max_abs_diff = float((g["sem"] - g["_sem_check"]).abs().max())

        if "n" in g.columns and len(g["n"]) > 0:
            n_values_all.extend(g["n"].tolist())

        x = g[pn].to_numpy()
        y = g["mean"].to_numpy()
        e = g["sem"].to_numpy()

        mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(e)
        x, y, e = x[mask], y[mask], e[mask]
        order = np.argsort(x)
        x, y, e = x[order], y[order], e[order]

        
        if x.size >= 4 and np.unique(x).size >= 4:
            xs = np.linspace(x.min(), x.max(), 300)
            ys = make_interp_spline(x, y, k=3)(xs)
            es = make_interp_spline(x, e, k=3)(xs)
        else:
            xs, ys, es = x, y, e

        color = shades[p]
        ls = line_styles[p]
        ax.plot(xs, ys, color=color, linewidth=2.2, linestyle=ls,
                label=param_labels_for_legend[p])
        ax.fill_between(xs, ys - es, ys + es, color=color, alpha=0.20)

        summary_lines.append(
            f"[AUDIT] {metric} vs {p} (normalized: {pn}) -> "
            f"rows={len(g)}, max|SEM - std/sqrt(n)|={max_abs_diff:.6g}, "
            f"CSV={audit_csv.name}"
        )

    ax.set_ylim(global_y_limits[metric])
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_ylabel(metric, fontsize=20, weight="bold", color=base_colors[metric])
    ax.set_xlabel("Normalized Parameter Value", fontsize=20, weight="bold")
    ax.tick_params(axis="both", labelsize=16, width=1.5)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight("bold")
        
    if metric in INCLUDE_LEGEND_FOR:    
        leg = ax.legend(frameon=False, fontsize=11, loc="best")
        for txt in leg.get_texts():
            txt.set_fontweight("bold")

    # Save plot
    fig.tight_layout()
    plot_path = out_dir / f"{metric.replace(' ', '_').lower()}.png"
    plt.savefig(plot_path, dpi=300)
    plt.close(fig)

    # Per-bin p-values across parameters for this metric
    pval_df = compute_per_bin_pvalues(df, metric_name=metric, num_bins=10)
    pval_csv = stats_dir / f"pvals_per_bin_{metric.replace(' ', '_').lower()}.csv"
    pval_df.to_csv(pval_csv, index=False)

    # Summarize n range and p-value info
    if n_values_all:
        n_min, n_max = int(np.min(n_values_all)), int(np.max(n_values_all))
        summary_lines.append(f"[INFO] {metric}: n per point ranges from {n_min} to {n_max}.")
    else:
        summary_lines.append(f"[INFO] {metric}: n per point not available (no groups).")
    summary_lines.append(f"[PVAL] Per-bin comparisons across PP/CE/MC saved to {pval_csv.name}.")
    summary_lines.append(f"[FIG ] Saved plot: {plot_path.name}")
    summary_lines.append("")

# -----------------------------
# Write summary text file 
# -----------------------------
summary_lines.insert(0, "CAPTION SUGGESTION:\n"
                        "Data are represented as mean ± SEM. Per-point sample size (n) ranges are listed below for each metric. "
                        "Per-bin comparisons among parameters were assessed by one-way ANOVA with Benjamini–Hochberg FDR correction "
                        "across bins (Kruskal–Wallis fallback if ANOVA was not applicable). Spline smoothing (cubic) is used for "
                        "visualization only; all statistics are computed on unsmoothed grouped values.\n")

with open(summary_txt_path, "w") as f:
    f.write("\n".join(summary_lines))

print("✅ Done.")
print(f"- Figures saved to: {out_dir}")
print(f"- Stats & CSVs saved to: {stats_dir}")
print(f"- Summary text file: {summary_txt_path}")

# -----------------------------
# Save a separate legend figure (black, no colors)
# -----------------------------
legend_order = ["Proliferative Probability", "Contact Energy", "Chemotaxis Lambda"]

handles = []
labels  = []
for key in legend_order:
    ls = line_styles[key]
    # black line sample with the right line style
    h = Line2D([0], [0], color="black", linestyle=ls, linewidth=2.5)
    handles.append(h)
    labels.append(param_labels_for_legend[key])

fig, ax = plt.subplots(figsize=(4.6, 2.0))
ax.axis("off")

leg = ax.legend(
    handles, labels,
    loc="center",
    frameon=True,
    framealpha=1.0,
    facecolor="white",
    edgecolor="black",
    fontsize=12,
)

# Optional: bolden legend text for consistency with your plots
for txt in leg.get_texts():
    txt.set_fontweight("bold")

fig.tight_layout()
legend_png = out_dir / "legend_parameters.png"
legend_pdf = out_dir / "legend_parameters.pdf"
plt.savefig(legend_png, dpi=300, bbox_inches="tight", pad_inches=0.25)
plt.savefig(legend_pdf,          bbox_inches="tight", pad_inches=0.25)
plt.close(fig)

print(f"- Separate legend saved to: {legend_png} and {legend_pdf}")


✅ Done.
- Figures saved to: ../../Results/Plots/Invasion_Metrics_Plots/Invasion_Metrics_Trends
- Stats & CSVs saved to: ../../Results/Plots/Invasion_Metrics_Plots/Invasion_Metrics_Trends/stats
- Summary text file: ../../Results/Plots/Invasion_Metrics_Plots/Invasion_Metrics_Trends/stats/figure_stats_summary.txt
- Separate legend saved to: ../../Results/Plots/Invasion_Metrics_Plots/Invasion_Metrics_Trends/legend_parameters.png and ../../Results/Plots/Invasion_Metrics_Plots/Invasion_Metrics_Trends/legend_parameters.pdf


## 2D Plots

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from matplotlib.gridspec import GridSpec


notebook_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))
combined_file_path = "../../Results/Data/invasion_metrics.csv"
base_output_path = "../../Results/Plots/Invasion_Metrics_Plots/2D_plot"

os.makedirs(base_output_path, exist_ok=True)
df_all = pd.read_csv(combined_file_path)

param_cols = ["Contact Energy", "Chemotaxis Lambda", "Proliferative Probability"]

metric_cols = ["Invasive Area", "Infiltrative Area", "Singles", "Fingers", "Clusters"]

agg_map = {col: "mean" for col in metric_cols}

df_mean = (
    df_all
    .groupby(param_cols, as_index=False)
    .agg(agg_map)
)


features = [
    ("Invasive Area", "Invasive Area"),
    ("Infiltrative Area", "Infiltrative Area"),
    ("Singles", "Singles"),
    ("Fingers", "Fingers"),
    ("Clusters", "Clusters")
]

# === Axis label mapping (for display) ===
axis_label_map = {
    "Chemotaxis Lambda": "Migration Coefficient",
    "Migration Coefficient": "Migration Coefficient"
}


def label_fix(label):
    return axis_label_map.get(label, label)

# === Main function to process and save row plots ===
def process_feature_2D_rows(df, feature_column, label_name):
    x = df["Contact Energy"].values
    y = df["Chemotaxis Lambda"].values
    z = df["Proliferative Probability"].values
    x_half_new = 0
    y_half_new = 15
    z_half = (z.max() + z.min()) / 2

    # Slicing configs
    plot_params = [
        ("Proliferative Probability", [z.min(), z_half, z.max()], "Contact Energy", "Chemotaxis Lambda"),
        ("Contact Energy", [x.min(), x_half_new, x.max()], "Chemotaxis Lambda", "Proliferative Probability"),
        ("Chemotaxis Lambda", [y.min(), y_half_new, y.max()], "Contact Energy", "Proliferative Probability")
    ]

    def compute_global_vrange(fixed_param, fixed_values, param1, param2):
        all_vals = []
        for val in fixed_values:
            mask = np.isclose(df[fixed_param], val, atol=0.1)
            subset = df[mask]
            if not subset.empty:
                all_vals.extend(subset[feature_column].values)
        return np.nanmin(all_vals), np.nanmax(all_vals)

    def plot_fixed_param_row(fixed_param, fixed_values, param1, param2, vmin, vmax):
        fig = plt.figure(figsize=(24, 8))
        gs = GridSpec(1, 4, width_ratios=[1, 1, 1, 0.05], wspace=0.3)
        axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
        # colorbar_ax = fig.add_subplot(gs[0, 3])
        # last_contour = None

        for i, val in enumerate(fixed_values):
            mask = np.isclose(df[fixed_param], val, atol=0.1)
            subset = df[mask]
            if subset.empty:
                continue
            X, Y = np.meshgrid(
                np.linspace(subset[param1].min(), subset[param1].max(), 100),
                np.linspace(subset[param2].min(), subset[param2].max(), 100)
            )
            Z = griddata((subset[param1], subset[param2]), subset[feature_column], (X, Y), method='cubic')
            if np.isnan(Z).all():
                Z = griddata((subset[param1], subset[param2]), subset[feature_column], (X, Y), method='nearest')
            last_contour = axes[i].contourf(X, Y, Z, levels=20, cmap='inferno', vmin=vmin, vmax=vmax)

            # Custom label shortening
            short_name_map = {
                "Proliferative Probability": "Proliferative Prob",
                "Chemotaxis Lambda": "Migration Coefficient",
                "Contact Energy": "Contact Energy"
            }
            short_label = short_name_map.get(fixed_param, fixed_param)
            val_formatted = f"{val:.1f}".rstrip('0').rstrip('.') if val % 1 else f"{int(val)}"
            axes[i].set_xlabel(label_fix(param1), fontsize=25, fontweight='bold')
            # axes[i].set_ylabel(label_fix(param2), fontsize=18, fontweight='bold')
            if i == 0:
                axes[i].set_ylabel(label_fix(param2), fontsize=26, fontweight='bold')
            else:
                axes[i].set_ylabel("")       
            axes[i].tick_params(axis='both', which='major', labelsize=24)
            for tick in axes[i].get_xticklabels():
                tick.set_fontweight('bold')
            for tick in axes[i].get_yticklabels():
                tick.set_fontweight('bold')
            # axes[i].set_title(f"{label_fix(fixed_param)} = {val:.2f}", fontsize=20, fontweight='bold')
            axes[i].set_title(f"{short_label} = {val_formatted}", fontsize=24, fontweight='bold')
                        

        metric_folder = os.path.join(base_output_path, feature_column.replace(" ", "_"))
        os.makedirs(metric_folder, exist_ok=True)

        filename = f"{label_fix(fixed_param).replace(' ', '_')}_row.png"
        output_path = os.path.join(metric_folder, filename)

        plt.tight_layout()
        plt.savefig(output_path, dpi=300)
        plt.close()
        return output_path

    output_paths = []
    for fixed_param, fixed_values, param1, param2 in plot_params:
        vmin, vmax = compute_global_vrange(fixed_param, fixed_values, param1, param2)
        output_paths.append(plot_fixed_param_row(fixed_param, fixed_values, param1, param2, vmin, vmax))

    return output_paths

results_rows = []
for feature_column, label in features:
    result_paths = process_feature_2D_rows(df_mean, feature_column, label)
    results_rows.extend(result_paths)



stats = (
    df_all
    .groupby(param_cols)
    .agg({m: ["mean", "std", "count"] for m in metric_cols})
)


for m in metric_cols:
    stats[(m, "sem")] = stats[(m, "std")] / np.sqrt(stats[(m, "count")])

stats.columns = [f"{a}_{b}" for a, b in stats.columns]
stats = stats.reset_index()
stats_csv_path = os.path.join(base_output_path, "2D_invasion_metrics_stats.csv")
stats.to_csv(stats_csv_path, index=False)


for path in results_rows:
    print("Saved:", path)


/var/folders/jr/3tw1qcls4879795w5933hkj00000gn/T/ipykernel_69201/1813259000.py:127: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Invasive_Area/Proliferative_Probability_row.png
Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Invasive_Area/Contact_Energy_row.png
Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Invasive_Area/Migration_Coefficient_row.png
Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Infiltrative_Area/Proliferative_Probability_row.png
Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Infiltrative_Area/Contact_Energy_row.png
Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Infiltrative_Area/Migration_Coefficient_row.png
Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Singles/Proliferative_Probability_row.png
Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Singles/Contact_Energy_row.png
Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Singles/Migration_Coefficient_row.png
Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Fingers/Proliferative_Probability_row.p

In [41]:
from PIL import Image
import os


metrics = {
    "Invasive_Area": [
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Invasive_Area/Proliferative_Probability_row.png",
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Invasive_Area/Contact_Energy_row.png",
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Invasive_Area/Migration_Coefficient_row.png",
    ],
    "Infiltrative_Area": [
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Infiltrative_Area/Proliferative_Probability_row.png",
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Infiltrative_Area/Contact_Energy_row.png",
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Infiltrative_Area/Migration_Coefficient_row.png",
    ],
    "Singles": [
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Singles/Proliferative_Probability_row.png",
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Singles/Contact_Energy_row.png",
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Singles/Migration_Coefficient_row.png",
    ],
    "Fingers": [
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Fingers/Proliferative_Probability_row.png",
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Fingers/Contact_Energy_row.png",
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Fingers/Migration_Coefficient_row.png",
    ],
    "Clusters": [
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Clusters/Proliferative_Probability_row.png",
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Clusters/Contact_Energy_row.png",
        "../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Clusters/Migration_Coefficient_row.png",
    ]
}

# === Stack rows vertically for each metric ===
for metric, paths in metrics.items():
    # Open images
    imgs = [Image.open(p) for p in paths]

    # Compute width and total height
    width = max(img.width for img in imgs)
    total_height = sum(img.height for img in imgs)

    # Create new blank canvas
    stacked = Image.new("RGB", (width, total_height))

    # Paste images
    y_offset = 0
    for im in imgs:
        stacked.paste(im, (0, y_offset))
        y_offset += im.height

    # Save stacked image
    out_path = f"../../Results/Plots/Invasion_Metrics_Plots/2D_plot/{metric}_2D_Stacked.png"
    stacked.save(out_path)
    print("Saved:", out_path)


Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Invasive_Area_2D_Stacked.png
Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Infiltrative_Area_2D_Stacked.png
Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Singles_2D_Stacked.png
Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Fingers_2D_Stacked.png
Saved: ../../Results/Plots/Invasion_Metrics_Plots/2D_plot/Clusters_2D_Stacked.png


## 3D plots

In [40]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
import matplotlib.cm as cm
import matplotlib.colors as mcolors


notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))




combined_file_path = "../../Results/Data/invasion_metrics.csv"
base_output_path = "../../Results/Plots/Invasion_Metrics_Plots/3D_plot"
os.makedirs(base_output_path, exist_ok=True)
df_all = pd.read_csv(combined_file_path)
param_cols = ["Contact Energy", "Chemotaxis Lambda", "Proliferative Probability"]
metric_cols = ["Invasive Area", "Infiltrative Area", "Singles", "Fingers", "Clusters"]
agg_map = {col: "mean" for col in metric_cols}

df_mean = (
    df_all
    .groupby(param_cols, as_index=False)
    .agg(agg_map)
)


features = [
    ("Invasive Area", "Invasive Area"),
    ("Infiltrative Area", "Infiltrative Area"),
    ("Singles", "Singles"),
    ("Fingers", "Fingers"),
    ("Clusters", "Clusters")
]

def process_feature_3D(df, feature_column, label_name):
    x = df["Contact Energy"].values
    y = df["Chemotaxis Lambda"].values
    z = df["Proliferative Probability"].values
    c = df[feature_column].values

    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()
    z_min, z_max = z.min(), z.max()

    X, Z = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(z_min, z_max, 100))
    Y, _ = np.meshgrid(np.linspace(y_min, y_max, 100), np.linspace(z_min, z_max, 100))
    X_top, Y_top = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))

    grid_points = np.array([x, y, z]).T
    C_left = griddata(grid_points, c, (X, np.full_like(X, y_max), Z), method='linear', fill_value=c.min())
    C_right = griddata(grid_points, c, (np.full_like(Y, x_max), Y, Z), method='linear', fill_value=c.min())
    C_top = griddata(grid_points, c, (X_top, Y_top, np.full_like(X_top, z_max)), method='linear', fill_value=c.min())

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    colormap = cm.inferno
    norm = mcolors.Normalize(vmin=c.min(), vmax=c.max())

    ax.plot_surface(X, np.full_like(X, y_max), Z, facecolors=colormap((C_left - c.min()) / (c.max() - c.min())), edgecolor='none', alpha=1.0)
    ax.plot_surface(np.full_like(Y, x_max), Y, Z, facecolors=colormap((C_right - c.min()) / (c.max() - c.min())), edgecolor='none', alpha=1.0)
    ax.plot_surface(X_top, Y_top, np.full_like(X_top, z_max), facecolors=colormap((C_top - c.min()) / (c.max() - c.min())), edgecolor='none', alpha=1.0)

    ax.set_xlabel("Contact Energy", fontsize=11, weight='bold')
    ax.set_ylabel("Migration Coefficient", fontsize=10, weight='bold')
    ax.set_zlabel("Proliferative Probability", fontsize=12, weight='bold')

    for tick in ax.xaxis.get_ticklabels(): tick.set_fontweight('bold')
    for tick in ax.yaxis.get_ticklabels(): tick.set_fontweight('bold')
    for tick in ax.zaxis.get_ticklabels(): tick.set_fontweight('bold')

    ax.view_init(elev=30, azim=40)
    cbar = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=colormap), ax=ax, shrink=0.8, aspect=20, pad=0.05)
    cbar.set_label(label_name, weight='bold', fontsize=18)
    cbar.ax.tick_params(labelsize=14)
    for tick in cbar.ax.get_yticklabels(): tick.set_fontweight('bold')

    safe_name = feature_column.replace(" ", "_")
    output_path = os.path.join(base_output_path, f"{safe_name}_3D_Cube.png")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()
    return output_path

# --- Run ---
results_3d = [process_feature_3D(df_mean, col, label) for col, label in features]
print("3D plots saved:", results_3d)


/var/folders/jr/3tw1qcls4879795w5933hkj00000gn/T/ipykernel_69201/2743440550.py:84: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


3D plots saved: ['../../Results/Plots/Invasion_Metrics_Plots/3D_plot/Invasive_Area_3D_Cube.png', '../../Results/Plots/Invasion_Metrics_Plots/3D_plot/Infiltrative_Area_3D_Cube.png', '../../Results/Plots/Invasion_Metrics_Plots/3D_plot/Singles_3D_Cube.png', '../../Results/Plots/Invasion_Metrics_Plots/3D_plot/Fingers_3D_Cube.png', '../../Results/Plots/Invasion_Metrics_Plots/3D_plot/Clusters_3D_Cube.png']


In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
import matplotlib.cm as cm
import matplotlib.colors as mcolors


notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))


combined_file_path = "../../Results/Data/invasion_metrics.csv"
base_output_path = "../../Results/Plots/Invasion_Metrics_Plots/3D_plot"
os.makedirs(base_output_path, exist_ok=True)
df_all = pd.read_csv(combined_file_path)
param_cols = ["Contact Energy", "Chemotaxis Lambda", "Proliferative Probability"]
metric_cols = ["Invasive Area", "Infiltrative Area", "Singles", "Fingers", "Clusters"]
agg_map = {col: "mean" for col in metric_cols}

df_mean = (
    df_all
    .groupby(param_cols, as_index=False)
    .agg(agg_map)
)


features = [
    ("Invasive Area", "Invasive Area"),
    ("Infiltrative Area", "Infiltrative Area"),
    ("Singles", "Singles"),
    ("Fingers", "Fingers"),
    ("Clusters", "Clusters")
]

def process_feature_3D(df, feature_column, label_name):
    x = df["Contact Energy"].to_numpy()
    y = df["Chemotaxis Lambda"].to_numpy()
    z = df["Proliferative Probability"].to_numpy()
    c = df[feature_column].to_numpy()

    x_min, x_max = np.min(x), np.max(x)
    y_min, y_max = np.min(y), np.max(y)
    z_min, z_max = np.min(z), np.max(z)

    # Faces of the cube
    X, Z = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(z_min, z_max, 100))
    Y, _ = np.meshgrid(np.linspace(y_min, y_max, 100), np.linspace(z_min, z_max, 100))
    X_top, Y_top = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))

    pts = np.column_stack([x, y, z])

    def interp_or_nearest(query):
        out = griddata(pts, c, query, method="linear")
        if np.isnan(out).any():
            out = griddata(pts, c, query, method="nearest")
        return out

    C_left  = interp_or_nearest((X, np.full_like(X, y_max), Z))          # y = max
    C_right = interp_or_nearest((np.full_like(Y, x_max), Y, Z))          # x = max
    C_top   = interp_or_nearest((X_top, Y_top, np.full_like(X_top, z_max)))  # z = max

    cmin, cmax = np.nanmin(c), np.nanmax(c)
    norm = mcolors.Normalize(vmin=cmin, vmax=cmax)
    cmap = cm.turbo

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    ax.plot_surface(X, np.full_like(X, y_max), Z,
                    facecolors=cmap(norm(C_left)), edgecolor="none", alpha=1.0)
    ax.plot_surface(np.full_like(Y, x_max), Y, Z,
                    facecolors=cmap(norm(C_right)), edgecolor="none", alpha=1.0)
    ax.plot_surface(X_top, Y_top, np.full_like(X_top, z_max),
                    facecolors=cmap(norm(C_top)), edgecolor="none", alpha=1.0)

    ax.set_xlabel("Contact Energy", fontsize=11, weight="bold")
    ax.set_ylabel("Migration Coefficient", fontsize=10, weight="bold")
    ax.set_zlabel("Proliferative Probability", fontsize=12, weight="bold")
    for tick in ax.xaxis.get_ticklabels(): tick.set_fontweight("bold")
    for tick in ax.yaxis.get_ticklabels(): tick.set_fontweight("bold")
    for tick in ax.zaxis.get_ticklabels(): tick.set_fontweight("bold")

    ax.view_init(elev=30, azim=40)

    cbar = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax,
                        shrink=0.8, aspect=20, pad=0.05)
    cbar.set_label(label_name, weight="bold", fontsize=18)
    cbar.ax.tick_params(labelsize=14)
    for tick in cbar.ax.get_yticklabels(): tick.set_fontweight("bold")

    safe_name = feature_column.replace(" ", "_")
    output_path = os.path.join(base_output_path, f"{safe_name}_3D_Cube.png")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()
    return output_path


# --- Run ---
results_3d = [process_feature_3D(df_mean, col, label) for col, label in features]
print("3D plots saved:", results_3d)


/var/folders/jr/3tw1qcls4879795w5933hkj00000gn/T/ipykernel_69201/408910721.py:98: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


3D plots saved: ['../../Results/Plots/Invasion_Metrics_Plots/3D_plot/Invasive_Area_3D_Cube.png', '../../Results/Plots/Invasion_Metrics_Plots/3D_plot/Infiltrative_Area_3D_Cube.png', '../../Results/Plots/Invasion_Metrics_Plots/3D_plot/Singles_3D_Cube.png', '../../Results/Plots/Invasion_Metrics_Plots/3D_plot/Fingers_3D_Cube.png', '../../Results/Plots/Invasion_Metrics_Plots/3D_plot/Clusters_3D_Cube.png']


## 2D + 3D Plots

In [43]:
from PIL import Image, ImageChops
import matplotlib.pyplot as plt
import os

# === Base Directories ===
base_dir = os.getcwd()
root_3d = os.path.abspath(os.path.join(base_dir, "../../Results/Plots/Invasion_Metrics_Plots/3D_plot"))
root_2d = os.path.abspath(os.path.join(base_dir, "../../Results/Plots/Invasion_Metrics_Plots/2D_plot"))
output_dir = os.path.abspath(os.path.join(base_dir, "../../Results/Plots/Invasion_Metrics_Plots/3D+2D_combined"))

# Ensure output folder exists
os.makedirs(output_dir, exist_ok=True)

# === Metrics and labels ===
metrics = [
    ("Clusters", "E"),
    ("Fingers", "D"),
    ("Infiltrative_Area", "B"),
    ("Invasive_Area", "A"),
    ("Singles", "C")
]

def trim_whitespace(image):
    bg = Image.new(image.mode, image.size, (255, 255, 255))
    diff = ImageChops.difference(image, bg)
    bbox = diff.getbbox()
    return image.crop(bbox) if bbox else image

for metric, label in metrics:
    # === Load and preprocess images ===
    img_3d = Image.open(os.path.join(root_3d, f"{metric}_3D_Cube.png")).convert("RGB")
    img_2d = Image.open(os.path.join(root_2d, f"{metric}_2D_Stacked.png")).convert("RGB")

    img_3d = trim_whitespace(img_3d)
    img_2d = trim_whitespace(img_2d)

    # === Resize 2D to match 3D height ===
    new_width_2d = int(img_2d.width * (img_3d.height / img_2d.height))
    img_2d_scaled = img_2d.resize((new_width_2d, img_3d.height))

    # === Combine images with slight overlap ===
    overlap = 40
    total_width = img_3d.width + img_2d_scaled.width + overlap
    combined_img = Image.new("RGB", (total_width, img_3d.height), color=(255, 255, 255))
    combined_img.paste(img_3d, (0, 0))
    combined_img.paste(img_2d_scaled, (img_3d.width + overlap, 0))

    # === Display and label ===
    plt.figure(figsize=(total_width / 100, img_3d.height / 100))
    plt.imshow(combined_img)
    plt.axis('off')

    label_text = f"{label}: {metric.replace('_', ' ')}"
    plt.text(10, 50, label_text, fontsize=90, fontweight='bold', color='black', backgroundcolor='white')

    # === Save figure ===
    out_path = os.path.join(output_dir, f"{label}_{metric}_Combined.png")
    plt.savefig(out_path, dpi=100, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {out_path}")


✅ Saved: /Users/sheriffakeeb/Desktop/TumorInvasion/Results/Plots/Invasion_Metrics_Plots/3D+2D_combined/E_Clusters_Combined.png
✅ Saved: /Users/sheriffakeeb/Desktop/TumorInvasion/Results/Plots/Invasion_Metrics_Plots/3D+2D_combined/D_Fingers_Combined.png
✅ Saved: /Users/sheriffakeeb/Desktop/TumorInvasion/Results/Plots/Invasion_Metrics_Plots/3D+2D_combined/B_Infiltrative_Area_Combined.png
✅ Saved: /Users/sheriffakeeb/Desktop/TumorInvasion/Results/Plots/Invasion_Metrics_Plots/3D+2D_combined/A_Invasive_Area_Combined.png
✅ Saved: /Users/sheriffakeeb/Desktop/TumorInvasion/Results/Plots/Invasion_Metrics_Plots/3D+2D_combined/C_Singles_Combined.png


In [46]:
from PIL import Image, ImageChops
import matplotlib.pyplot as plt
import os

# === Base Directories ===
base_dir = os.getcwd()
root_3d = os.path.abspath(os.path.join(base_dir, "../../Results/Plots/Invasion_Metrics_Plots/3D_plot"))
root_2d = os.path.abspath(os.path.join(base_dir, "../../Results/Plots/Invasion_Metrics_Plots/2D_plot"))
output_dir = os.path.abspath(os.path.join(base_dir, "../../Results/Plots/Invasion_Metrics_Plots/3D+2D_combined"))

# Ensure output folder exists
os.makedirs(output_dir, exist_ok=True)

# === Metrics and labels ===
metrics = [
    ("Clusters", "E"),
    ("Fingers", "D"),
    ("Infiltrative_Area", "B"),
    ("Invasive_Area", "A"),
    ("Singles", "C")
]

# === Options ===
include_label = False       
use_letter_label = False   

def trim_whitespace(image):
    bg = Image.new(image.mode, image.size, (255, 255, 255))
    diff = ImageChops.difference(image, bg)
    bbox = diff.getbbox()
    return image.crop(bbox) if bbox else image

for metric, letter in metrics:

    img_3d = Image.open(os.path.join(root_3d, f"{metric}_3D_Cube.png")).convert("RGB")
    img_2d = Image.open(os.path.join(root_2d, f"{metric}_2D_Stacked.png")).convert("RGB")

    img_3d = trim_whitespace(img_3d)
    img_2d = trim_whitespace(img_2d)

    new_width_2d = int(img_2d.width * (img_3d.height / img_2d.height))
    img_2d_scaled = img_2d.resize((new_width_2d, img_3d.height))


    overlap = 40
    total_width = img_3d.width + img_2d_scaled.width + overlap
    combined_img = Image.new("RGB", (total_width, img_3d.height), color=(255, 255, 255))
    combined_img.paste(img_3d, (0, 0))
    combined_img.paste(img_2d_scaled, (img_3d.width + overlap, 0))


    plt.figure(figsize=(total_width / 100, img_3d.height / 100))
    plt.imshow(combined_img)
    plt.axis('off')

    if include_label:
        if use_letter_label:
            label_text = f"{letter}: {metric.replace('_', ' ')}"
        else:
            label_text = metric.replace("_", " ")

        plt.text(10, 50, label_text, fontsize=90, fontweight='bold',
                 color='black', backgroundcolor='white')

    out_path = os.path.join(output_dir, f"{metric}_Combined.png")
    plt.savefig(out_path, dpi=100, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {out_path}")


✅ Saved: /Users/sheriffakeeb/Desktop/TumorInvasion/Results/Plots/Invasion_Metrics_Plots/3D+2D_combined/Clusters_Combined.png
✅ Saved: /Users/sheriffakeeb/Desktop/TumorInvasion/Results/Plots/Invasion_Metrics_Plots/3D+2D_combined/Fingers_Combined.png
✅ Saved: /Users/sheriffakeeb/Desktop/TumorInvasion/Results/Plots/Invasion_Metrics_Plots/3D+2D_combined/Infiltrative_Area_Combined.png
✅ Saved: /Users/sheriffakeeb/Desktop/TumorInvasion/Results/Plots/Invasion_Metrics_Plots/3D+2D_combined/Invasive_Area_Combined.png
✅ Saved: /Users/sheriffakeeb/Desktop/TumorInvasion/Results/Plots/Invasion_Metrics_Plots/3D+2D_combined/Singles_Combined.png
